# EcoGraphRAG — Notebook 5: Ablation (Graph-Only)

**Goal:** Run graph-only retrieval (no FAISS) + Mistral-7B. Proves that graph alone isn't enough — the combination is what works.

**Outputs:** `ablation_results.json`

## 1. Setup

In [ ]:
from google.colab import drive
import os, json, time, re, pickle
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/graphrag_research/'
DRIVE_DATA = DRIVE_ROOT+'data/'; DRIVE_INDICES = DRIVE_ROOT+'indices/'
DRIVE_OUTPUTS = DRIVE_ROOT+'outputs/'; DRIVE_CHECKPOINTS = DRIVE_ROOT+'checkpoints/'
for d in [DRIVE_OUTPUTS,DRIVE_CHECKPOINTS]: os.makedirs(d,exist_ok=True)
print('Drive mounted.')

In [ ]:
!pip install -q transformers accelerate bitsandbytes spacy networkx
!python -m spacy download en_core_web_sm -q

## 2. Load Data

In [ ]:
import torch, spacy
from collections import defaultdict, deque
with open(DRIVE_DATA+'hotpotqa_500.json') as f: questions=json.load(f)['questions']
with open(DRIVE_DATA+'chunks.json') as f: chunks=json.load(f)
chunk_lookup = {c['chunk_id']:c for c in chunks}
with open(DRIVE_INDICES+'entity_graph.gpickle','rb') as f: graph=pickle.load(f)
nlp = spacy.load('en_core_web_sm',disable=['tagger','parser','lemmatizer'])
print(f'Qs:{len(questions)} Chunks:{len(chunks)} Graph:{graph.number_of_nodes()}n/{graph.number_of_edges()}e')

## 3. Graph-Only Retrieval

In [ ]:
GRAPH_TOP_K = 6
BFS_DEPTH = 2

def extract_query_entities(question):
    doc = nlp(question)
    seen,ents = set(),[]
    for e in doc.ents:
        k=e.text.strip().lower()
        if len(k)>=2 and k not in seen: seen.add(k); ents.append(k)
    return ents

def graph_only_retrieve(question, top_k=GRAPH_TOP_K):
    q_ents = extract_query_entities(question)
    visited, cs, nt = set(), defaultdict(float), 0
    for ent in q_ents:
        if ent not in graph: continue
        q, lv = deque([(ent,0)]), set()
        while q:
            nd,d = q.popleft()
            if nd in lv: continue
            lv.add(nd)
            if nd not in visited:
                visited.add(nd); nt += 1
                for cid in graph.nodes[nd].get('chunk_ids',[]):
                    cs[cid] += 1.0/(1.0+d)
            if d < BFS_DEPTH:
                for nb in graph.neighbors(nd):
                    if nb not in lv: q.append((nb,d+1))
    ranked = sorted(cs.items(), key=lambda x:x[1], reverse=True)
    result_chunks = []
    for cid,_ in ranked[:top_k]:
        if cid in chunk_lookup: result_chunks.append(chunk_lookup[cid])
    return result_chunks, q_ents, nt

# Test
tc,te,tn = graph_only_retrieve('Were Scott Derrickson and Ed Wood of the same nationality?')
print(f'Test: {len(tc)} chunks, ents={te}, nodes={tn}')
for c in tc: print(f'  {c["chunk_id"]}|{c["title"]}')

## 4. Load Mistral-7B

In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
LLM_MODEL='mistralai/Mistral-7B-Instruct-v0.2'
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
tokenizer=AutoTokenizer.from_pretrained(LLM_MODEL)
model=AutoModelForCausalLM.from_pretrained(LLM_MODEL,quantization_config=bnb,device_map='auto',torch_dtype=torch.float16)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
print(f'Model loaded. GPU:{torch.cuda.max_memory_allocated()/(1024**2):.0f}MB')

In [ ]:
PROMPT='<s>[INST] Answer the following question using ONLY the provided context.\nGive a short, precise answer (1-5 words). Do not explain.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer: [/INST]'
def generate_answer(question,ctx):
    p=PROMPT.format(context='\n\n'.join([c['text'] for c in ctx]),question=question)
    inp=tokenizer(p,return_tensors='pt',truncation=True,max_length=2048)
    inp={k:v.to(model.device) for k,v in inp.items()}
    with torch.no_grad(): out=model.generate(**inp,max_new_tokens=64,do_sample=False,temperature=0.1,pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:],skip_special_tokens=True).strip()

## 5. Run Ablation (500 Qs)

In [ ]:
import tracemalloc
CKPT=DRIVE_CHECKPOINTS+'ablation_checkpoint.json'
RFILE=DRIVE_OUTPUTS+'ablation_results.json'
if os.path.exists(CKPT):
    with open(CKPT) as f: ck=json.load(f)
    results,si = ck['results'],ck['current_index']+1
else: results,si = [],0
total=len(questions)
print(f'Running ablation (graph-only): {si} to {total-1}...\n')
for i in range(si,total):
    q=questions[i]
    torch.cuda.reset_peak_memory_stats(); tracemalloc.start(); t0=time.time()
    ctx,qe,nt = graph_only_retrieve(q['question'])
    pred = generate_answer(q['question'],ctx)
    lat=time.time()-t0; gpu=torch.cuda.max_memory_allocated()/(1024**2)
    _,ram=tracemalloc.get_traced_memory(); tracemalloc.stop()
    results.append({'id':q['id'],'question':q['question'],'prediction':pred,'gold':q['answer'],'type':q['type'],'level':q.get('level','unknown'),'latency_seconds':round(lat,3),'peak_gpu_mb':round(gpu,2),'peak_ram_mb':round(ram/(1024**2),2),'chunks_retrieved':len(ctx),'nodes_traversed':nt,'query_entities':qe})
    if (i+1)%10==0 or i==total-1:
        print(f'  [{i+1}/{total}] {lat:.2f}s nodes={nt} Pred:"{pred[:35]}" Gold:"{q["answer"]}"')
    if (i+1)%25==0:
        with open(CKPT,'w') as f: json.dump({'current_index':i,'results':results},f,ensure_ascii=False)
        print(f'  Checkpoint saved')
with open(RFILE,'w') as f: json.dump(results,f,indent=2,ensure_ascii=False)
print(f'\nSaved {len(results)} results')
if os.path.exists(CKPT): os.remove(CKPT)

## 6. Evaluation

In [ ]:
import string
from collections import Counter
def normalize(s):
    s=s.lower(); s=re.sub(r'\b(a|an|the)\b',' ',s); s=''.join(c for c in s if c not in string.punctuation); return ' '.join(s.split())
def em(p,g): return int(normalize(p)==normalize(g))
def f1(p,g):
    pt,gt=normalize(p).split(),normalize(g).split()
    if not pt or not gt: return float(pt==gt)
    c=sum((Counter(pt)&Counter(gt)).values())
    if c==0: return 0.0
    return 2*(c/len(pt))*(c/len(gt))/(c/len(pt)+c/len(gt))
em_s=[em(r['prediction'],r['gold']) for r in results]
f1_s=[f1(r['prediction'],r['gold']) for r in results]
print(f'\n{"="*60}\nABLATION: GRAPH-ONLY ({len(results)} Qs)\n{"="*60}')
print(f'Overall EM: {sum(em_s)/len(em_s):.4f} ({sum(em_s)/len(em_s)*100:.1f}%)')
print(f'Overall F1: {sum(f1_s)/len(f1_s):.4f} ({sum(f1_s)/len(f1_s)*100:.1f}%)')
for qt in ['bridge','comparison']:
    tr=[r for r in results if r['type']==qt]
    if tr:
        te=[em(r['prediction'],r['gold']) for r in tr]; tf=[f1(r['prediction'],r['gold']) for r in tr]
        print(f'{qt.capitalize()} ({len(tr)}): EM:{sum(te)/len(te):.4f} F1:{sum(tf)/len(tf):.4f}')
nl=[r['nodes_traversed'] for r in results]; la=[r['latency_seconds'] for r in results]
print(f'\nNodes: avg={sum(nl)/len(nl):.0f} max={max(nl)}')
print(f'Latency: avg={sum(la)/len(la):.2f}s')
metrics={'experiment':'ablation_graph_only','num_questions':len(results),'overall_em':round(sum(em_s)/len(em_s),4),'overall_f1':round(sum(f1_s)/len(f1_s),4),'avg_latency_s':round(sum(la)/len(la),3),'avg_nodes_traversed':round(sum(nl)/len(nl),1)}
with open(DRIVE_OUTPUTS+'ablation_metrics.json','w') as f: json.dump(metrics,f,indent=2)
print('\nMetrics saved. Ablation complete!')